## <center><span style="color:blue">Fla</span><span style="color:gold">sk</span> UI for <span style="color:blue">Gemini</span> Resume Optimizer</span></center>
***
#### Where the <span style="color:gold">Gradio</span> UI was used for the <span style="color:orange">OpenAI</span> / <span style="color:orange">ChatGPT</span> optimizer, the following notebook does the same for our <span style="color:blue">Gemini</span> optimizer.
***
### Step 1: Imports
> #### <u><span style="color:red">Standard imports as other notebooks, only this time we're importing the functions from the Python file ('<span style="color:firebrick">gemini_flash_functions</span>'), as well as <span style="color:green">Flask</span> and its following libraries</u></span>:
> ##### 1.) "<span style="color:green">Flask</span>" - to create the web app object;
> ##### 2.) "<span style="color:green">render_template</span>" - renders HTML templates (this is a webapp, after all);
> ##### 3.) "<span style="color:green">request</span>" - deals with the job description inputs that the users copy and paste into the program; and
> ##### 4.) "<span style="color:green">flash</span>" - creates temporary, one-time messages for user interaction;
> ##### 5.) "<span style="color:green">redirect</span>" - redirects user to an actual URL so they don't have to reload the webapp after every interaction;
> ##### 6.) "<span style="color:green">url_for</span>" - converts functions to actual URL paths inside the <span style="color:green">Flask</span> application; and 
> ##### 7.) "<span style="color:green">send_file</span>" - sends files to the user's browser(s).

#### One other added import: '<span style="color:green">tempfile</span>'. '<span style="color:green">Template</span>' is a <span style="color:blue">Python</span> module that creates temporary files and directories. It's used to:
> ##### a.) create an actual temporary Markdown file - the downloaded resume - to the browser; and
> ##### b.) it sets up a temporary path to that file and writes the optimized resume to that file.

In [1]:
from flask import Flask, render_template, request, flash, redirect, url_for, send_file # jsonify(?)
import os
from dotenv import load_dotenv
import google.generativeai as genai
from markdown import markdown
import tempfile

# import the functions from 'gemini_flash_functions' python file:
from gemini_flask_functions import (
    configure_gemini_api,
    tailored_resume_function_schema,
    gemini_initialization_with_function_calling,
    generate_tailoring_prompt,
    get_gemini_response_with_function_calling
)

In [2]:
# start Flask
app = Flask(__name__)

In [3]:
# Load environment variables
load_dotenv()

True

### <span style="color:red">Ruh-roh.</span>
#### <span style="color:green">Flask</span> starts up fine as the webapp, but keeps erroring out on a Markdown filter not being found whenever we try to actually use it.
#### After some homework, it looks like that's because <span style="color:green">Flask</span> uses <span style="color:firebrick">Jinja2</span> for its HTML templating, and <span style="color:firebrick">Jinja2</span> doesn't have any sort of Markdown filter. So, we have to make the conversion in the <span style="color:green">Flask</span> route itself.
> ##### Reviewing the code below, the Markdown <i>was</i> converted to HTML (the '<span style="color:hotpink">tailored_resume_html</span> = <span style="color:blue">markdown</span>(<span style="color:forestgreen">tailored_resume</span>)' line), so the problem may rest in the '<b>templates</b>' directory (in <span style="color:blue">VS Code</span>).
> ##### After running the app in the app in its virtual environment, <span style="color:steelblue">Mac Terminal</span> hit me with some more details, signalling an error in the 'Additional Suggestions' result.

In [4]:
# call the api
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")
if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not loaded from your environment.")
genai.configure(api_key=GEMINI_API_KEY)

In [5]:
# # app = Flask(__name__)
# # app.secret_key = os.environ.get("SECRET_KEY", "dev-secret-key")  # Change this in production

# from flask import Flask, render_template, request, flash, redirect, url_for, send_file
# import os
# import tempfile
# from dotenv import load_dotenv
# from markdown import markdown

# # Import functions from the functions_for_gemini_flask module
# from functions_for_gemini_flask import (
#     configure_gemini_api,
#     tailored_resume_function_schema,
#     gemini_initialization_with_function_calling,
#     generate_tailoring_prompt,
#     get_gemini_response_with_function_calling
# )

# # Load environment variables
# load_dotenv()

# app = Flask(__name__)
# app.secret_key = os.environ.get("SECRET_KEY", "dev-secret-key")  # Change this in production

def parse_gemini_response(response):
    """
    Parse the Gemini API response to extract the tailored resume and suggestions.
    This is an adaptation of the handle_gemini_response function that returns values
    instead of writing to a file.
    """
    tailored_resume = None
    additional_suggestions = None
    
    if response.candidates:
        first_candidate = response.candidates[0]

        if first_candidate.content and first_candidate.content.parts:
            first_part = first_candidate.content.parts[0]

            if first_part.function_call:
                function_call = first_part.function_call

                if function_call.name == "tailor_resume":
                    arguments = function_call.args
                    tailored_resume = arguments.get("tailored_resume")
                    additional_suggestions = arguments.get("additional_suggestions")
            elif first_part.text:
                # Fallback if function calling didn't work
                tailored_resume = first_part.text
    
    return tailored_resume, additional_suggestions

# Configure Gemini API on app startup
try:
    configure_gemini_api()
    function_schema = tailored_resume_function_schema()
    gemini_model = gemini_initialization_with_function_calling(function_schema)
except ValueError as e:
    print(f"Error during initialization: {e}")
    gemini_model = None

@app.route("/", methods=["GET"])
def index():
    """Home page with form to upload resume and job description"""
    return render_template('index.html')
    
@app.route("/tailor-resume", methods=["GET", "POST"])
def tailor_resume():
    """Process the resume and job description to generate a tailored resume"""
    if not gemini_model:
        flash("API configuration error. Please check your environment variables.")
        return redirect(url_for("index"))
    
    # Get resume and job description from form
    resume_text = request.form.get("resume_text")
    jd_text = request.form.get("job_description")
    
    if not resume_text or not jd_text:
        flash("Please provide both resume and job description.")
        return redirect(url_for("index"))
    
    try:
        # Generate tailored resume
        prompt = generate_tailoring_prompt(resume_text, jd_text)
        response = get_gemini_response_with_function_calling(
            gemini_model, 
            prompt, 
            function_schema
        )
        
        tailored_resume, additional_suggestions = parse_gemini_response(response)
        
        if not tailored_resume:
            flash("Failed to generate tailored resume. Please try again.")
            return redirect(url_for("index"))
            
        # Convert Tailored Resume Markdown to HTML for display
        tailored_resume_html = markdown(tailored_resume)
        additional_suggestions_html = markdown(additional_suggestions)

        # Convert Additional Suggestions Markdow to HTML for display
        return render_template(
            "result.html", 
            tailored_resume_html = tailored_resume_html,
            tailored_resume_md = tailored_resume,
            additional_suggestions_html = additional_suggestions_html
        )
        
    except Exception as e:
        flash(f"An error occurred: {str(e)}")
        return redirect(url_for("index"))

@app.route("/download-resume", methods=["POST"])
def download_resume():
    """Download the tailored resume as a markdown file"""
    tailored_resume = request.form.get("tailored_resume_md")
    
    if not tailored_resume:
        flash("No resume data to download.")
        return redirect(url_for("index"))
    
    # Create a temporary file
    with tempfile.NamedTemporaryFile(delete=False, suffix=".md") as temp:
        temp_path = temp.name
        temp.write(tailored_resume.encode("utf-8"))
    
    # Send the file to the user
    try:
        return send_file(
            temp_path,
            as_attachment = True,
            download_name = "tailored_resume.md",
            mimetype = "text/markdown"
        )
    finally:
        # Clean up the temporary file after sending
        os.unlink(temp_path)

if __name__ == "__main__":
    # Check if running in IPython/Jupyter
    try:
        get_ipython
        # If we're in IPython/Jupyter, don't use the auto-reloader
        print("Flask app is ready. Access it at http://127.0.0.1:5000/")
        print("To run the app, use: app.run()")
    except NameError:
        # Normal Python environment
        app.run(debug = True)

Flask app is ready. Access it at http://127.0.0.1:5000/
To run the app, use: app.run()


In [ ]:
app.run()

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
